# Baseline — Zero-Shot Prompting (no training)

Prompts the untrained base model with the same system instruction, relevant snippet and query used by every trained method, and decodes an answer directly. No parameters are updated; this measures what supplying the context buys on its own.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip -q install -U datasets huggingface_hub transformers accelerate bitsandbytes rouge-score

    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSET_NAME = "all_subset"
SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"

RESULTS_DIR = PP_ROOT / SUBSET_NAME / "v2_personamem_zero_shot_50_snippet"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Subset:", SUBSET_NAME)
print("Results will be saved to:", RESULTS_DIR.resolve())

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 219.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 68.1 MB/s eta 0:00:00
Mounted at /content/drive
Subset: all_subset
Results will be saved to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_zero_shot_50_snippet


In [2]:
import ast
import json
import random
from typing import Any, Dict, List

import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

DATASET_NAME = "bowen-upenn/PersonaMem-v2"
CONFIG = "benchmark"
TEXT_SPLITS = ("train_text", "val_text", "benchmark_text")

SEED = 42
TOKENIZER_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
random.seed(SEED)


def load_split(split):
    if split not in TEXT_SPLITS:
        raise ValueError(f"split must be one of {TEXT_SPLITS}, got {split!r}")
    return load_dataset(DATASET_NAME, CONFIG, split=split)


def parse_user_query(raw):
    if isinstance(raw, dict):
        return str(raw.get("content", raw)).strip()
    if isinstance(raw, str):
        try:
            d = ast.literal_eval(raw)
            if isinstance(d, dict):
                return str(d.get("content", raw)).strip()
        except (ValueError, SyntaxError):
            pass
        return raw.strip()
    return str(raw).strip()


def parse_incorrect_answers(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if hasattr(raw, "tolist"):
        return [str(x) for x in raw.tolist()]
    if isinstance(raw, str):
        try:
            val = ast.literal_eval(raw)
            if isinstance(val, list):
                return [str(x) for x in val]
        except (ValueError, SyntaxError):
            pass
        return [raw]
    return []

def get_snippet(row):
    val = row.get("related_conversation_snippet")
    if val is None:
        return ""
    return str(val).strip()

In [3]:
def load_subset(name, subsets_dir=SUBSETS_DIR):
    """Load a persona subset (ids + train/val rows) saved by create_persona_subsets.ipynb."""
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"Subset not found: {subset_dir}. Run create_persona_subsets.ipynb first.")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = [int(p) for p in json.load(f)]
    tr = pd.read_parquet(subset_dir / "train.parquet")
    va = pd.read_parquet(subset_dir / "val.parquet")
    return sorted(persona_ids), tr, va

MIN_VAL_ROWS = 4
CLIENT_PERSONAS, train_df, val_df = load_subset(SUBSET_NAME)
NUM_CLIENTS = len(CLIENT_PERSONAS)
val_counts = val_df.groupby("persona_id").size()
print(f"train rows: {len(train_df):,} | val rows: {len(val_df):,}")
print(f"Subset '{SUBSET_NAME}': {NUM_CLIENTS} personas")
print(f"Personas ({len(CLIENT_PERSONAS)}): first 10 = {CLIENT_PERSONAS[:10]} ...")
print(f"Val rows per selected persona (min/mean/max): "
      f"{val_counts[CLIENT_PERSONAS].min()}/{val_counts[CLIENT_PERSONAS].mean():.1f}/{val_counts[CLIENT_PERSONAS].max()}")

train rows: 3,870 | val rows: 714
Subset 'all_subset': 150 personas
Personas (150): first 10 = [6, 9, 14, 18, 33, 41, 51, 56, 57, 73] ...
Val rows per selected persona (min/mean/max): 4/4.8/8


In [4]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SNIPPET_TOKENS = 1024
MAX_NEW_TOKENS = 256
EVAL_BATCH_SIZE = 32
EVAL_TRAIN = False

model_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token
model_tok.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    trust_remote_code=True, attn_implementation="sdpa",
)
model.config.use_cache = True
model.eval()

print("Loaded zero-shot model:", MODEL_NAME)


def build_system_prompt():
    return (
        "You are a personalised assistant. Use the conversation snippet to find "
        "information or connections relevant to the question, then provide the answer."
    )

def truncate_snippet(snippet, max_tokens):
    ids = model_tok(snippet, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return snippet
    return model_tok.decode(ids[-max_tokens:], skip_special_tokens=True)


def build_user_content(row):
    snippet = truncate_snippet(get_snippet(row), MAX_SNIPPET_TOKENS)
    return (
        "Relevant snippet from our earlier conversation:\n"
        f"{snippet}\n\n"
        f"{parse_user_query(row['user_query'])}"
    )


client_data: Dict[Any, Dict[str, Any]] = {}
for pid in CLIENT_PERSONAS:
    p_train = train_df[train_df["persona_id"] == pid].reset_index(drop=True)
    system_prompt = build_system_prompt()
    client_data[pid] = {"system_prompt": system_prompt}

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded zero-shot model: Qwen/Qwen2.5-1.5B-Instruct
